In [22]:
import os
# os.environ["PYTORCH_CUDA_ALLOC_CONF"] = (
#     "backend:cudaMallocAsync,garbage_collection_threshold:0.5,"
#     "max_split_size_mb:32,expandable_segments:True"
# )

import torch
import torch.nn as nn
pushforwardtype = "fromprior"
if pushforwardtype =="fromjoint":
    from FNOmodel import *
    Cin=3
elif pushforwardtype=="fromprior":
    from FNOmodel_2inputchannel import *
    Cin=2
import numpy as np
import os
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
# from utilities import MatReader 
import h5py
import time

# Loading data

In [23]:
# loading the data
file_path = "./../../Data/Experiment7/samples_wave_center.npz"
f=np.load(file_path)


In [24]:
Cout=1
width=100
modes_x=32
depth=5 

BATCH_SIZE  = 80
SEED        = 42
Nepochs     = 15


use_cm=True
cm_alpha = f['alpha'].item()
cm_tau =   f['tau'].item()
cm_scale = f['scale'].item()

dropout=0
 
Nx=101
Lx=1.0
dx=Lx/(Nx-1)
sigma_noise=5e-3

 
# loading data:
x= f["U"]
y= f["arrival"]
np.random.seed(42)
y_noisy = y + np.random.normal(0, sigma_noise, size=y.shape)

print("y_noisy shape:", y_noisy.shape)
print("x shape:", x.shape)
print("y shape:", y.shape)
print("x dtype:", x.dtype)
print("y dtype:", y.dtype)

y_noisy shape: (10, 2000000)
x shape: (100, 2000000)
y shape: (10, 2000000)
x dtype: float64
y dtype: float64


In [25]:
device = "cuda" if torch.cuda.is_available() else "cpu"
# print_memory("after loading libs")


In [26]:
n_total = len(x[0,:])


In [27]:
n_total

2000000

In [28]:
# ----- config -----
n_total = len(x[0,:])
learning_ratio = 9/12 # compared to experiment 4, in experiment 5 I generated 1.2 *1e6 data so if you want 1e6 data points, you need the ratio ot be 10/12
TEST_SIZE=0.05
n_subset = int(learning_ratio * n_total)
 

rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)

# ----- keep only first half -----
x_sub = x[:, 1:n_subset+1]          # (N,N,n_subset-1) 
y_sub = y_noisy[:, 1:n_subset+1]    # (8,8,n_subset-1)

# ----- move sample axis front for sklearn/pytorch: (N,H,W) -----
x_sub = np.moveaxis(x_sub, -1, 0).astype(np.float32)  # (10000,64)
y_sub = np.moveaxis(y_sub, -1, 0).astype(np.float32)  # (10000,  8)

# ----- split train / test within first half -----
x_tr, x_te, y_tr, y_te = train_test_split(
    x_sub, y_sub, test_size=TEST_SIZE, random_state=SEED, shuffle=True
)

# ----- to torch tensors and flatten -----
x_tr_t = torch.from_numpy(x_tr).view(x_tr.shape[0], -1)   # (N_tr, 201)
y_tr_t = torch.from_numpy(y_tr).view(y_tr.shape[0], -1)   # (N_tr,    201)

x_te_t = torch.from_numpy(x_te).view(x_te.shape[0], -1)   # (N_te, 201)
y_te_t = torch.from_numpy(y_te).view(y_te.shape[0], -1)   # (N_te,    10)



In [31]:
n_subset

1333333

In [32]:
class IndexedJointDataset(torch.utils.data.Dataset):
    def __init__(self, u, y):
        self.u = u.clone().detach().float()
        self.y = y.clone().detach().float()

    def __len__(self):
        return len(self.u)

    def __getitem__(self, idx):
        return self.u[idx], self.y[idx], idx


In [33]:
train_dataset = IndexedJointDataset(x_tr_t, y_tr_t)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_dataset = torch.utils.data.TensorDataset(x_te_t, y_te_t)
val_loader  = torch.utils.data.DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=True)

In [34]:

x_tr_t_mean = x_tr_t.mean().item()
x_tr_t_std  = x_tr_t.std().item()
y_tr_t_mean = y_tr_t.mean().item()
y_tr_t_std  = y_tr_t.std().item()

mean_std = {
    'mean_u': x_tr_t_mean,
    'std_u':  x_tr_t_std,
    'mean_y': y_tr_t_mean,
    'std_y':  y_tr_t_std,
}



In [35]:
# print_memory("after loading data")


# Defining FNO model

In [36]:
use_cm

True

In [37]:
base = CosineFNO1d(
    Cin=Cin, Cout=Cout,
    width=width, modes_x=modes_x,  Lx=Lx,
    depth=depth, use_cm=use_cm, cm_alpha=cm_alpha, cm_tau=cm_tau, cm_scale=cm_scale, dropout=dropout)


In [38]:
if pushforwardtype =="fromjoint":
    model = FNOTripletAdapter1d(
    base,
    mean_std['mean_u'], mean_std['std_u'],
    mean_std['mean_y'], mean_std['std_y']).to(device)
elif pushforwardtype=="fromprior":
    model = FNO2Adapter1d(
    base,
    mean_std['mean_u'], mean_std['std_u'],
    mean_std['mean_y'], mean_std['std_y']).to(device)

# wrap for multi-GPU if available
if torch.cuda.device_count() > 1:
    print("Using", torch.cuda.device_count(), "GPUs!")
    model = nn.DataParallel(model)

print(model)


FNO2Adapter1d(
  (fields): MultiInputCosineFNO1d(
    (fno): CosineFNO1d(
      (lift): Conv1d(2, 100, kernel_size=(1,), stride=(1,))
      (blocks): ModuleList(
        (0-5): 6 x CosineFNOblock1d(
          (spec): CosineSpectralConv1d()
          (w_pt): Conv1d(100, 100, kernel_size=(1,), stride=(1,))
          (act): GELU(approximate='none')
          (do): Identity()
        )
      )
      (proj): Conv1d(100, 1, kernel_size=(1,), stride=(1,))
      (cm): CMProjectionNeumann1d()
    )
  )
)


In [39]:
# print_memory("after loading model")


## testing the data loading and model defenition

In [40]:


# model = model.to(device)

# # get one batch
# u_vec, y_vec, idx = next(iter(train_loader))

# # for now, use ydag = y (can change later if needed)
# ydag_vec = y_vec.clone()

# # move to device
# u_vec, y_vec, ydag_vec = u_vec.to(device), y_vec.to(device), ydag_vec.to(device)

# # forward pass
# with torch.no_grad():
#     out = model(u_vec, y_vec, ydag_vec)

# print("u_vec shape   :", u_vec.shape)    # expected (B, 65536)
# print("y_vec shape   :", y_vec.shape)    # expected (B,    64)
# print("ydag_vec shape:", ydag_vec.shape) # expected (B,    64)
# print("out shape     :", out.shape)      # expected (B, 65536)


# Defining loss

In [41]:
def energy_distance_loss(model, u_batch, y_batch, Nx=100):
    
    """
    Computes the empirical energy-distance-based loss:
        L(θ) = Accuracy Term - Repulsion Term

    Args:
        model      : neural network implementing T(u, y; y^dag) u \in \R^latent_dim, y \in R^n_obs,  
        u_batch    : Tensor of shape (B,\R^latent_dim), batch of u samples
        y_batch    : Tensor of shape (B,\R^n_obs), corresponding y values
        Potential inputs for upscaling:
        indices    : Tensor of shape (B,), indices of the batch samples in the full dataset
        full_u     : Tensor of shape (N,), all training u values
        full_y     : Tensor of shape (N,), all training y values

    Returns:
        Scalar loss (torch.float32)
    """
    B = u_batch.shape[0] # Batch size
    latent_dim= u_batch.shape[1] # Latent dimension
    n_obs = y_batch.shape[1]  # Number of observations

    dx = 1.0 / Nx
    cell_area_sqrt = (dx) ** 0.5  # multiply norms by this factor

    # using vectorization to avoid loops:
    # Generate all (i, j) pairs for accuracy term,
    u_i = u_batch.unsqueeze(1).expand(B, B ,latent_dim).reshape(-1, latent_dim)
    y_i = y_batch.unsqueeze(1).expand(B, B ,n_obs).reshape(-1, n_obs)
    y_j = y_batch.unsqueeze(0).expand(B, B ,n_obs).reshape(-1, n_obs)
    u_j = u_batch.unsqueeze(0).expand(B, B ,latent_dim).reshape(-1, latent_dim)

    acc_pred = model(u_i, y_i, y_j)
    acc_loss = torch.mean(torch.norm(acc_pred - u_j,dim=-1))* cell_area_sqrt # l^2 loss

    # === Repulsion term ===
    # For each y_k (k=0,...,B-1), we compute T(u_i, y_i; y_k)
    # Shape: (B, B)
    y_k_matrix = y_batch.view(1, B, n_obs).expand(B, B, n_obs)  # shape (B, B, n_obs)
    u_i_matrix = u_batch.view(B, 1, latent_dim).expand(B, B, latent_dim)  # shape (B, B, latent_dim)
    y_i_matrix = y_batch.view(B, 1, n_obs).expand(B, B, n_obs)  # shape (B, B, n_obs)

    u_i_flat = u_i_matrix.reshape(-1, latent_dim)
    y_i_flat = y_i_matrix.reshape(-1, n_obs)
    y_k_flat = y_k_matrix.reshape(-1, n_obs)



    # Evaluate model: shape (B, B, latent_dim)
    T_out = model(u_i_flat, y_i_flat, y_k_flat)  # [B², latent_dim]
    T_out = T_out.view(B, B, latent_dim)         # [B, B, latent_dim]

    # Compute pairwise distances between rows: |T_i - T_j| for each y_k (column)
    # Result: matrix of shape (B, B) for each column → stack into (B, B, B)
    diffs = T_out.unsqueeze(0) - T_out.unsqueeze(1)  # shape (B, B, B)
    dists = torch.norm(diffs,dim=-1)* cell_area_sqrt  # pairwise ||T_i - T_j|| for each fixed y_k

    # Average over all (i, j) including diagonals, then correct
    repulsion_loss = dists.mean() * (B / (B - 1))
    # print("u_i:", u_i)
    # print("y_i:", y_i)
    # print("y_j:", y_j)
    # print("u_j:", u_j)
    # print("pred:", acc_loss)
    # print("repulsion_loss:", repulsion_loss)
    return 2 * acc_loss - repulsion_loss


In [42]:
# import torch

# def energy_distance_loss_fromprior(model, u_batch, y_batch, Nx=100):
#     """
#     Same loss as your energy_distance_loss_fromprior, but avoids ALL in-place writes
#     (no fill_diagonal_ / no dists[...] = 0).
#     """
#     B = u_batch.shape[0]
#     latent_dim = u_batch.shape[1]
#     n_obs = y_batch.shape[1]

#     dx = 1.0 / Nx
#     cell_area_sqrt = dx

#     # ---------- Accuracy term ----------
#     u_i = u_batch.unsqueeze(1).expand(B, B, latent_dim).reshape(-1, latent_dim)
#     y_j = y_batch.unsqueeze(0).expand(B, B, n_obs).reshape(-1, n_obs)
#     u_j = u_batch.unsqueeze(0).expand(B, B, latent_dim).reshape(-1, latent_dim)

#     acc_pred = model(u_i, y_j)  # [B^2, latent_dim]
#     acc_matrix = torch.norm(acc_pred - u_j, dim=-1).view(B, B)  # [B,B]

#     # mask off diagonal (no inplace)
#     off_diag = (~torch.eye(B, device=acc_matrix.device, dtype=torch.bool)).to(acc_matrix.dtype)
#     acc_loss = (acc_matrix * off_diag).sum() / (B * (B - 1)) * cell_area_sqrt

#     # ---------- Repulsion term ----------
#     y_k_matrix = y_batch.view(1, B, n_obs).expand(B, B, n_obs)          # [B,B,n_obs]
#     u_i_matrix = u_batch.view(B, 1, latent_dim).expand(B, B, latent_dim) # [B,B,latent_dim]

#     u_i_flat = u_i_matrix.reshape(-1, latent_dim)  # [B^2, latent_dim]
#     y_k_flat = y_k_matrix.reshape(-1, n_obs)       # [B^2, n_obs]

#     T_out = model(u_i_flat, y_k_flat).view(B, B, latent_dim)  # [B,B,latent_dim]

#     # diffs: [B,B,B,latent_dim] (pairwise across i/j, for each k implicitly as last axis of T_out)
#     diffs = T_out.unsqueeze(0) - T_out.unsqueeze(1)
#     dists = torch.norm(diffs, dim=-1) * cell_area_sqrt  # [B,B,B]

#     # build valid mask to zero out:
#     # 1) i=j (diagonal for each k)
#     # 2) i=k
#     # 3) j=k
#     idx = torch.arange(B, device=dists.device)
#     I = idx.view(B, 1, 1)
#     J = idx.view(1, B, 1)
#     K = idx.view(1, 1, B)

#     valid = (I != J) & (I != K) & (J != K)          # [B,B,B]
#     d_valid = dists * valid.to(dists.dtype)         # masked, no inplace

#     # Same as your: repulsion_loss = dists.mean() * (B / (B - 1))
#     # (We keep your exact normalization.)
#     repulsion_loss = d_valid.mean() * (B / (B - 1))

#     return 2 * acc_loss - repulsion_loss

In [ ]:
import torch

def energy_distance_loss_fromprior(model, u_batch, y_batch, Nx=100):
    """
    Same loss as your energy_distance_loss_fromprior, but avoids ALL in-place writes
    (no fill_diagonal_ / no dists[...] = 0).
    """
    B = u_batch.shape[0]
    latent_dim = u_batch.shape[1]
    n_obs = y_batch.shape[1]

    dx = 1.0 / Nx
    cell_area_sqrt = dx

    # ---------- Accuracy term ----------
    u_i = u_batch.unsqueeze(1).expand(B, B, latent_dim).reshape(-1, latent_dim)
    y_j = y_batch.unsqueeze(0).expand(B, B, n_obs).reshape(-1, n_obs)
    u_j = u_batch.unsqueeze(0).expand(B, B, latent_dim).reshape(-1, latent_dim)

    acc_pred = model(u_i, y_j)  # [B^2, latent_dim]
    acc_matrix = torch.norm(acc_pred - u_j, dim=-1).view(B, B)  # [B,B]

    # mask off diagonal (no inplace)
    off_diag = (~torch.eye(B, device=acc_matrix.device, dtype=torch.bool)).to(acc_matrix.dtype)
    acc_loss = (acc_matrix * off_diag).sum() / (B * (B - 1)) * cell_area_sqrt

    # ---------- Repulsion term ----------
    y_k_matrix = y_batch.view(1, B, n_obs).expand(B, B, n_obs)          # [B,B,n_obs]
    u_i_matrix = u_batch.view(B, 1, latent_dim).expand(B, B, latent_dim) # [B,B,latent_dim]

    u_i_flat = u_i_matrix.reshape(-1, latent_dim)  # [B^2, latent_dim]
    y_k_flat = y_k_matrix.reshape(-1, n_obs)       # [B^2, n_obs]

    T_out = model(u_i_flat, y_k_flat).view(B, B, latent_dim)  # [B,B,latent_dim]

    # diffs: [B,B,B,latent_dim] (pairwise across i/j, for each k implicitly as last axis of T_out)
    diffs = T_out.unsqueeze(0) - T_out.unsqueeze(1)
    dists = torch.norm(diffs, dim=-1) * cell_area_sqrt  # [B,B,B]

    # build valid mask to zero out:
    # 1) i=j (diagonal for each k)
    # 2) i=k
    # 3) j=k
    idx = torch.arange(B, device=dists.device)
    I = idx.view(B, 1, 1)
    J = idx.view(1, B, 1)
    K = idx.view(1, 1, B)

    valid = (I != J) & (I != K) & (J != K)          # [B,B,B]
    d_valid = dists * valid.to(dists.dtype)         # masked, no inplace

    repulsion_loss = d_valid.mean() * (B**3 / (B*(B - 1)*(B-2)))

    return 2 * acc_loss - repulsion_loss 

# Learning loop

In [43]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.3e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, verbose=True
)

epochs = Nepochs
train_losses = []
val_losses = []
# print_memory("after doing the loss")

/resnick/groups/astuart/hkaveh/soft/miniconda3/envs/chaoslearn/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:60: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [ ]:
# Training loop with validation loss
# Before training loop (once at top of script)
# torch.autograd.set_detect_anomaly(True)  # <-- only for debugging; 



# ----- Early stopping variables -----
best_val_loss = float('inf')
trigger_times = 0
best_model_state = None    # <--- store in RAM
patience = 70   # or any number you want
# ------------------------------------

for epoch in range(Nepochs):
    start_time = time.time()   # start timer

    model.train()
    total_train_loss = 0.0

    # Training pass
    for u_batch, y_batch, idx_batch in train_loader:
        u_batch = u_batch.to(device)
        y_batch = y_batch.to(device)
 
        optimizer.zero_grad()
        if pushforwardtype =="fromjoint":
            loss = energy_distance_loss(model, u_batch, y_batch)
        elif pushforwardtype=="fromprior":
            loss = energy_distance_loss_fromprior(model, u_batch, y_batch)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        if torch.isfinite(loss):
            optimizer.step()
        else:
            print(f"⚠️ Non-finite loss detected at epoch {epoch+1}, skipping optimizer step.")

        total_train_loss += loss.item()

    # Validation pass
    model.eval()
    with torch.no_grad():
        val_loss_total = 0.0
        val_batches = 0
        for u_val, y_val in val_loader:
            u_val = u_val.to(device)
            y_val = y_val.to(device)
            if pushforwardtype =="fromjoint":
                batch_loss = energy_distance_loss(model, u_val, y_val)
            elif pushforwardtype=="fromprior":
                batch_loss = energy_distance_loss_fromprior(model, u_val, y_val)
            val_loss_total += batch_loss.item()
            val_batches += 1
        val_loss = val_loss_total / val_batches
    scheduler.step(val_loss)

    # Logging
    end_time = time.time()
    epoch_time = end_time - start_time
    mins, secs = divmod(epoch_time, 60)

    print(f"Time: {int(mins)}m {secs:.1f}s")
    train_losses.append(total_train_loss / len(train_loader))
    val_losses.append(val_loss)
    print(f"Epoch {epoch+1}/{epochs} — Train Loss: {train_losses[-1]:.4f} — Val Loss: {val_losses[-1]:.4f}")
    print("lr =", optimizer.param_groups[0]["lr"])

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        trigger_times = 0
        # Save raw state dict (may include "module.")
        best_model_state = {
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
        }
    else:
        trigger_times += 1
        print("trigger_times is: "+str(trigger_times))
        if trigger_times >= patience:
            print("Early stopping triggered.")
            break


    # ---------------------------------


In [ ]:
# print_memory("after running")


In [ ]:
def plot_losses(train_losses, val_losses):
    """
    Plots training and validation loss curves.

    Args:
        train_losses (list of float): Training loss per epoch
        val_losses (list of float): Test loss per epoch
    """
    plt.figure(figsize=(8, 5))
    plt.plot(train_losses, label='Train Loss', marker='o')
    plt.plot(val_losses, label='Test Loss', marker='s')
    plt.yscale('log')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Test Loss')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_losses(train_losses, val_losses)

In [11]:
# # If model is wrapped in DataParallel, use .module
# model_to_save = model.module if isinstance(model, nn.DataParallel) else model

# torch.save({
#     'model_state_dict': model_to_save.state_dict(),
#     'mean_std': mean_std,   # also save normalization stats
# }, "./../../Models/fno_model_checkpoint.pth")

# print("Model saved successfully!")


Model saved successfully!


In [ ]:
# --- get base model even if wrapped in DataParallel ---
from datetime import datetime
# Restore best model before saving final checkpoint

# --- clean best_model_state if it contains DataParallel prefixes ---
def strip_module_prefix(state_dict):
    out = {}
    for k, v in state_dict.items():
        if k.startswith("module."):
            out[k[len("module."):]] = v
        else:
            out[k] = v
    return out

model_to_save = model.module if isinstance(model, nn.DataParallel) else model

if best_model_state is not None:
    best_model_state = strip_module_prefix(best_model_state)
    model_to_save.load_state_dict(best_model_state)

# --- metadata for file naming ---
model_type      = "CosineFNO"
cm_flag         = "cmON" if use_cm else "cmOFF"
dataset_tag     = "200_1D_1e6"
timestamp       = datetime.now().strftime("%Y%m%d_%H%M")
usemoredata     = ""
# --- build descriptive filename ---
save_name = (
    f"{model_type}_{cm_flag}_{pushforwardtype}_"
    f"B{BATCH_SIZE}_lr1e-3_E{Nepochs}_s{SEED}_depth{depth}_width_{width}_modes_x{modes_x}_{dataset_tag}withlrdecay_{timestamp}.pth"
)

# --- directory and full path ---
save_dir = "./../../Models/Experiment7/"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, save_name)

# --- build checkpoint ---
checkpoint = {
    'model_state_dict': model_to_save.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'mean_std': mean_std,
    'config': {
        'model_type': model_type,
        'use_cm': use_cm,
        'cm_alpha': getattr(base.cm, 'alpha', None),
        'cm_tau': getattr(base.cm, 'tau', None),
        'cm_scale': getattr(base.cm, 'scale', None),
        'width': width,
        'modes_x': modes_x,
        'depth': depth,
        'dropout': dropout,
        'epochs': Nepochs,
        'batch_size': BATCH_SIZE,
        'seed': SEED,
        'pushforwardtype': pushforwardtype,
        'dataset': dataset_tag,
        'timestamp': timestamp,
    },
    'train_losses': train_losses,
    'val_losses': val_losses,
}

# --- save everything ---
torch.save(checkpoint, save_path)
print(f"✅ Model saved successfully at:\n{save_path}")